In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
import os
os.environ["DIRECTORY_PATH"] = os.getenv("DIRECTORY_PATH")

In [4]:
from langchain_google_genai import ChatGoogleGenerativeAI
model = ChatGoogleGenerativeAI(model = "gemini-3.6-flash")
output = model.invoke("hi")
output.content

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.
Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


[{'type': 'text',
  'text': 'Hello! How can I help you today?',
  'extras': {'signature': 'Eo8GCowGAWkUfRO6EVCsv/uSTr7cYOmb3tzvdYMujyh9Ga4TtBj7yOw6qiy3E+HGiRznLW519hkTMYLCxWT59cd742cw58ztPuox0qnBtz8b9loeT6KALkFNAxTAAWjYDDL1bkGOJ332nIU0b2IPyIM9ayUQ3qckSbXH+LH8ZiPMh+x/kLMiRCOIOn4KI1JleHw/fAO9US3Xya+WyTksMuRJWsIR0LmfEUfbHB16ycILNvhSWbUiDbG1fVo1L+gEWAWuW8BO+2BGG5iqRd/edmCCl8jbnSnTfQMJ73l5kN93Jrg83dXa5RSz6a5K/Yyrt2sTOVi5JXC6XYiF3tWXxK2x2OYNMntmArIGFhcWttTOHWjmpJIh7FZQb9Hcogb8qSLeCXM7BmE4BZCy1Zwuvk5wdC28n2BwljWUhQA48Y21oV/xvP21pyR4xXIxkBOp3jXp1RNiEaQCHFX3VvEf5RkaNrgWqqw5hUyEE91ufsSD7qqoXLLvFVKbe0M1N1701xpPDHoZHzmwKXCWTIjpf4jEQo7ryNkkYdH51O69P3E7QgfwwdX7hTuuCKIZhZKQgXfhiFBvZRs1zG0odwFAkGKERfR1Vudn700R/cL79jAikSQCsvBFxfTskurzdG+q4kkn7aX/+y0I5/1DFo3aywsPkYM/4zgaeRIzoxzw30MW3zNdmUafTqWBOUxBAHwmeDeJbnP+WG7JqUtVexv3JQaM/Ux6Y32c1KnScPhtueFE1fJeL/SFo/CVc/9nZF0dOhbz1ypZBvj33+LAh3oPrXe1ZIrlxkmTCgdi5gIFcryNtO9uWPbJlTQbEaQaWq8ASuFxxF30F+Iv3VMlX1gZP6H7/Prqng/X5WpDb/tLTc0TVqMm9Hv2MeOgqBFZvrYxyG8AlRNVCYXnZ

In [5]:

from langchain_google_genai import GoogleGenerativeAIEmbeddings
embeddings = GoogleGenerativeAIEmbeddings(model = "gemini-embedding-001")
len(embeddings.embed_query("Hi"))

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


3072

In [6]:
from langchain_community.document_loaders import TextLoader, DirectoryLoader
from langchain_community.vectorstores import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter

C:\Users\Mohit\AppData\Local\Temp\ipykernel_20696\4068926105.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader, DirectoryLoader
d:\03_Study\GenAI\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
loader = DirectoryLoader(os.environ["DIRECTORY_PATH"] + "\\Source Files", glob= "./*.txt", loader_cls= TextLoader)

In [8]:
docs = loader.load()

In [9]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 200,
    chunk_overlap = 50
)

In [10]:
new_docs = text_splitter.split_documents(docs)

In [11]:
doc_chunks = [doc.page_content for doc in new_docs]

In [12]:
len(doc_chunks)

55

In [13]:
db = Chroma.from_documents(new_docs, embeddings)

In [14]:
retriever = db.as_retriever(search_kwargs = {"k" : 2})

In [15]:
retriever.invoke("Industrial Growth of US?")

[Document(metadata={'source': 'D:\\03_Study\\GenAI\\3.3 Traditional RAG 02\\Source Files\\usa.txt'}, page_content='The U.S. maintains its GDP growth through strong innovation, entrepreneurship, and investment in R&D. With companies like Apple, Google, Amazon, Microsoft, and Tesla leading global markets, the U.S.'),
 Document(metadata={'source': 'D:\\03_Study\\GenAI\\3.3 Traditional RAG 02\\Source Files\\usa.txt'}, page_content='Historically, the U.S. economy has enjoyed consistent long-term growth, averaging around 2-3% annually. Post-pandemic, the economy bounced back strongly, but 2022 and 2023 saw rising inflation due to')]

In [16]:
from pydantic import BaseModel, Field
class TopicSelectionParser(BaseModel):
    Topic:str = Field("Selected Topic")
    Reasoning:str = Field("Reason of selecting Topic")


In [21]:
from langchain_core.output_parsers import PydanticOutputParser
from typing import TypedDict, Annotated, Sequence
from langchain_core.messages import BaseMessage
import operator


In [18]:
parser = PydanticOutputParser(pydantic_object = TopicSelectionParser)

In [19]:
parser.get_format_instructions()

'The output should be formatted as a JSON instance that conforms to the JSON schema below.\n\nAs an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}\nthe object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.\n\nHere is the output schema:\n```\n{"properties": {"Topic": {"default": "Selected Topic", "title": "Topic", "type": "string"}, "Reasoning": {"default": "Reason of selecting Topic", "title": "Reasoning", "type": "string"}}}\n```'

In [ ]:
class AgentState(TypedDict):
    message: Annotated[Sequence[BaseMessage],operator.add]